# 03_Modeling.ipynb

## **0. Giới thiệu**

Notebook này thực hiện:

- Phân tích dữ liệu sau khi đã preprocessing.

- Thực hiện các kiểm định thống kê cơ bản.

- Áp dụng mô hình Machine Learning đơn giản (Logistic Regression bằng NumPy).

- Đánh giá mô hình bằng accuracy, precision, recall, confusion matrix.

- Vẽ biểu đồ từ thư viện `src/visualization.py`.

## **1. Import & Load dữ liệu đã tiền xử lý**

In [2]:
import numpy as np
import sys, os


# thêm đường dẫn để import src
project_root = os.path.abspath("..")
sys.path.append(project_root)


from src.visualization import (
    plot_hist_numeric,
    plot_top_categories,
    plot_pie,
    plot_scatter,
    plot_corr_heatmap
)


# load dữ liệu sạch từ file 02
X = np.load("../data/processed/data_clean.npy", allow_pickle=True)
headers = np.load("../data/processed/headers_clean.npy", allow_pickle=True)

## **2. Xác định biến đầu vào & target**

In [3]:
target_idx = headers.tolist().index("target")
y = X[:, target_idx].astype(float)


# loại bỏ target ra khỏi features
X_features = np.delete(X, target_idx, axis=1).astype(float)
print("Shape X:", X_features.shape)
print("Shape y:", y.shape)

Shape X: (19158, 14)
Shape y: (19158,)


## **3. Train-test split**

In [4]:
np.random.seed(0)
indices = np.random.permutation(len(X_features))
cut = int(0.8 * len(X_features))
train_idx, test_idx = indices[:cut], indices[cut:]


X_train, X_test = X_features[train_idx], X_features[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

## **4. Logistic Regression (tự cài bằng NumPy)**

Hàm sigmoid ổn định số học

In [5]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -20, 20)))

Hàm loss

In [6]:
def loss_fn(X, y, w):
    z = X @ w
    p = sigmoid(z)
    eps = 1e-9
    return -np.mean(y*np.log(p+eps) + (1-y)*np.log(1-p+eps))

Gradient descent

In [ ]:
def logistic_regression(X, y, lr=0.01, epochs=300):
    n_features = X.shape[1]
    w = np.zeros(n_features)


    for _ in range(epochs):
        z = X @ w
        p = sigmoid(z)
        grad = X.T @ (p - y) / len(y)
        w -= lr * grad
    return w

: 

Train

In [ ]:
w = logistic_regression(X_train, y_train)
print("Trained weights shape:", w.shape)

## **5. Dự đoán & Đánh giá mô hình**

In [ ]:
def predict(X, w, threshold=0.5):
    return (sigmoid(X @ w) >= threshold).astype(int)


train_pred = predict(X_train, w)
test_pred = predict(X_test, w)


train_acc = (train_pred == y_train).mean()
test_acc = (test_pred == y_test).mean()


print("Accuracy (train):", train_acc)
print("Accuracy (test): ", test_acc)

## **6. Confusion Matrix**

In [ ]:
def confusion_matrix(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[TP, FP],[FN, TN]])


cm = confusion_matrix(y_test, test_pred)
print("Confusion matrix:\n", cm)

## **7. Visualization trên dữ liệu sạch**

Histogram cho một số biến

In [ ]:
idx = headers.tolist().index("training_hours")
arr = X[:, idx].astype(float)
plot_hist_numeric(arr, title="Training Hours (scaled)")

Scatter đơn giản

In [ ]:
cdi = X[:, headers.tolist().index("city_development_index")].astype(float)
train = X[:, headers.tolist().index("training_hours")].astype(float)
plot_scatter(cdi, train, xlabel="CDI", ylabel="Training Hours")

Heatmap tương quan

In [ ]:
numeric_cols = [h for h in headers if h != "target"]
num_data = X.astype(float)
plot_corr_heatmap(num_data, numeric_cols)

## **8. Kiểm định giả thuyết thống kê (t-test ví dụ)**

Kiểm định: ứng viên muốn đổi việc có training_hours cao hơn không?

In [ ]:
train_0 = X[:, headers.tolist().index("training_hours")][y == 0]
train_1 = X[:, headers.tolist().index("training_hours")][y == 1]


mean0, mean1 = train_0.mean(), train_1.mean()
var0, var1 = train_0.var(), train_1.var()
n0, n1 = len(train_0), len(train_1)


# t-statistic (vectorized)
t_stat = (mean0 - mean1) / np.sqrt(var0/n0 + var1/n1)
print("t-statistic:", t_stat)

## **9. Kết luận**

Dữ liệu sau preprocessing hoạt động tốt với mô hình Logistic Regression.

Accuracy ổn nhưng có thể cải thiện bằng việc:

- tuning learning rate / epochs

- thêm regularization

- xử lý imbalance

- enrich thêm feature

Các bước visualization cho thấy feature interaction có ý nghĩa.

Kiểm định thống kê hỗ trợ việc hiểu sâu hành vi của ứng viên.